In [ ]:
import numpy as np
import pandas as pd
import pydicom
import os
import matplotlib.pyplot as plt
import collections
from tqdm import tqdm_notebook as tqdm
from datetime import datetime

from math import ceil, floor
import cv2

import tensorflow as tf
import keras

import sys

# from keras_applications.resnet import ResNet50
from keras_applications.inception_v3 import InceptionV3

from sklearn.model_selection import ShuffleSplit

import numpy as np
import pandas as pd
import pydicom
from pydicom.data import get_testdata_files
import os
from os import listdir
from os.path import isfile, join
from glob import glob
from random import sample
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

import cv2
from sklearn.model_selection import KFold
import collections
from tqdm import tqdm_notebook as tqdm
from datetime import datetime

from math import ceil, floor


import tensorflow as tf
import keras

import sys

from keras_applications.resnet import ResNet50

from sklearn.model_selection import ShuffleSplit


import seaborn as sns
sns.set()

from os import listdir

from skimage.transform import resize
import scipy.ndimage
from skimage import morphology
from skimage import measure
from skimage.transform import resize
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from plotly import __version__
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
from plotly.tools import FigureFactory as FF
from plotly.graph_objs import *

from sklearn.model_selection import train_test_split

from keras.applications import ResNet50, VGG16
from keras.applications.resnet50 import preprocess_input as preprocess_resnet_50
from keras.applications.vgg16 import preprocess_input as preprocess_vgg_16
from keras.layers import GlobalAveragePooling2D, Dense, Activation
from keras.models import Model
from keras.utils import Sequence
from keras import metrics
from keras.layers import Dense
from keras.layers import Conv2D

from keras.layers import BatchNormalization

#from tensorflow.keras.applications.resnet50 import preprocess_input
#from tensorflow.keras.applications import ResNet50
#from tensorflow.keras.preprocessing.image import load_img, img_to_array

from keras.models import Sequential
from keras.layers import Convolution2D

from keras.layers.normalization import BatchNormalization
from keras.layers import MaxPooling2D
from keras.layers import Flatten
from keras.layers import Dense

from keras.layers import *
from keras.models import Sequential
from keras.applications.resnet50 import ResNet50

from keras_applications.inception_v3 import InceptionV3

from sklearn.model_selection import ShuffleSplit


from keras.utils import to_categorical
from keras.metrics import categorical_accuracy

from keras.utils import np_utils

from keras import optimizers

import tqdm

from skimage.filters.rank import median
from skimage.morphology import disk

test_images_dir = '../input/rsna-intracranial-hemorrhage-detection/stage_2_test_images/'
#train_images_dir = '../input/rsna-intracranial-hemorrhage-detection/stage_2_train_images/'

In [ ]:
from skimage import data, color
from skimage.transform import rescale, resize, downscale_local_mean

prct_epid = 0.02
prct_intrap = 0.25
prct_intrav = 0.17
prct_subar = 0.25
prct_subdu = 0.31



def sample_by_distr_subtype(prct, size_of_samples, subtype,df):
    tr_type_1 = df[df[subtype] == 1]
    tr_type_0 = df[df[subtype] == 0]
    tr_type = pd.concat([tr_type_1.sample(int(size_of_samples*prct/2)),tr_type_0.sample(int(size_of_samples*prct/2))])
    return tr_type


def sample_by_distribution(size_of_samples,df):
    list_subtypes = list(df.columns)

    tr_epidural = sample_by_distr_subtype( prct_epid, size_of_samples,'epidural' ,df)
    tr_intrap = sample_by_distr_subtype( prct_intrap , size_of_samples,'intraparenchymal' ,df)
    tr_intrav = sample_by_distr_subtype( prct_intrav, size_of_samples,'intraventricular' ,df)
    tr_subar = sample_by_distr_subtype( prct_subar, size_of_samples,'subarachnoid' ,df)
    tr_subdu = sample_by_distr_subtype( prct_subdu, size_of_samples,'subdural' ,df)
    tr_any = df[df["any"] == 0].sample(int(size_of_samples * 0.72))
    
    return pd.concat([tr_epidural, tr_intrap, tr_intrav, tr_subar, tr_subdu, tr_any])


def get_data_samples(size_of_samples, df):
    #sample_df = pd.concat([train_df[train_df['Label'] == 1].sample(size_of_samples),train_df[train_df['Label'] == 0].sample(size_of_samples)])
    sample_df = sample_by_distribution(size_of_samples,df)
    xtrain, xval = train_test_split(sample_df,
                            test_size=0.1,
                            shuffle = True,
                            stratify=sample_df.values)
                            #random_state=1)
    return xtrain,xval

image_size = 224
def read_and_prep_images(img_paths, img_size=(224,224), pixels_out=True, folder=train_images_dir, display_mode=0):
    
    img_array = []
    gauge = tqdm.tqdm(range(len(img_paths))) if len(img_paths) > 1 else range(len(img_paths))
    
    for i in gauge:
        img_array.append(resize(apply_mask(get_pixels_hu(img_paths[i], folder=folder),-150,2000,display_mode=display_mode),(img_size[0],img_size[1])))

    img_array = np.array(img_array)

    mean = np.mean(np.array(img_array))
    std = np.std(np.array(img_array))
    std = 1 if std == 0 else std
    
    for i in range(0,img_array.shape[0]):
#         std = np.std(img_array[i])
#         std = 1 if std == 0 else std
        img_array[i] = (img_array[i] - mean) / std
    
    if pixels_out:
        img_array = np.repeat(img_array[..., np.newaxis], 3, -1)
    output = img_array
    #img_array = np.array([cv2.resize(img.pixel_array,(224,224)) for img in imgs])

    #output = preprocess_input(img_array)
    
    return output

def get_ds(img_name, folder=train_images_dir):
    pydicom_filedataset = pydicom.read_file(os.path.join(folder, img_name))
    return pydicom_filedataset

def get_pixel_array(img_name, folder=train_images_dir):
    pydicom_filedataset = pydicom.read_file(os.path.join(folder, img_name))
    return pydicom_filedataset.pixel_array

def get_pixels_hu(img_name, folder=train_images_dir):
    try:
        image = get_pixel_array(img_name, folder=folder)
        dataset = get_ds(img_name, folder=folder)
        # Convert to int16 (from sometimes int16), 
        # should be possible as values should always be low enough (<32k)
        image= image.astype(np.int16)

        # Set outside-of-scan pixels to 1
        # The intercept is usually -1024, so air is approximately 0
        image[image <= -2000] = 0

        # Convert to Hounsfield units (HU)
        intercept = dataset.RescaleIntercept
        slope = dataset.RescaleSlope

        if slope != 1:
            image = slope * image.astype(np.float64)
            image = image.astype(np.int16)

        image += np.int16(intercept)
        return np.array(image, dtype=np.int16)
    except:
        print("File {} is corrupted, removing from set".format(img_name))
        return False

def set_manual_window(hu_image, custom_center, custom_width):
    min_value = custom_center - (custom_width/2)
    max_value = custom_center + (custom_width/2)
    hu_image[hu_image < min_value] = min_value
    hu_image[hu_image > max_value] = max_value
    return hu_image


def view_image(img):
    plt.imshow(get_pixel_array(img), cmap='bone')

def apply_mask(img,th,upper_th,display_mode=0):
    print('starting apply mask')
    if img is False:            # In case get_pixels_hu crashed, we need to return an empty image
        return np.array([[0]], dtype='float32')
    row_size= img.shape[0]
    col_size = img.shape[1]
    
    #thresh_img = np.where((img - 1000) / 150 < 1,1.0,0.0)  # threshold the image
    filter_hu_bounds = (img>th) & (img<upper_th)
    thresh_img = np.where(filter_hu_bounds,1.0,0.0)  # threshold the image
    
    
    #eroded = morphology.erosion(thresh_img,np.ones([3,3]))
    #dilation = morphology.dilation(eroded,np.ones([8,8]))
    
    dilation = thresh_img #median(thresh_img, disk(10))

    labels = measure.label(dilation, background= -1) # Different labels are displayed in different colors
    regions = measure.regionprops(labels,img)
    regions.sort(reverse=True, key=lambda x:x.bbox_area)
    regions.remove(regions[0])
    regions.sort(reverse=True, key=lambda x:x.area)
    
    if (len(regions) <= 0):
        return np.array([[0]], dtype='float32')
    skull_region=regions[0]
    x,y,x1,y1=skull_region.bbox
    cropped_image = img[x:x1,y:y1]
    cropped_image_filled = np.where(skull_region.filled_image, cropped_image, -100000 * np.ones(cropped_image.shape))
    mask = np.where(cropped_image_filled>-100000,1,0)
    
    if display_mode == 1:
        fig, ax = plt.subplots(3, 2, figsize=[12, 12])
        ax[0, 0].set_title("Original")
        ax[0, 0].imshow(img, cmap='gray')
        ax[0, 0].axis('off')
        ax[0, 1].set_title("Threshold")
        ax[0, 1].imshow(thresh_img, cmap='gray')
        ax[0, 1].axis('off')
        ax[1, 0].set_title("After Erosion and Dilation")
        ax[1, 0].imshow(dilation, cmap='gray')
        ax[1, 0].axis('off')
        ax[1, 1].set_title("Color Labels")
        ax[1, 1].imshow(labels)
        ax[1, 1].axis('off')
        ax[2, 0].set_title("Final Mask")
        ax[2, 0].imshow(mask, cmap='gray')
        ax[2, 0].axis('off')
        ax[2, 1].set_title("Apply Mask on Original")
        ax[2, 1].imshow(cropped_image * mask, cmap='gray')
        ax[2, 1].axis('off')
        plt.show()

    if display_mode == 2:
        fig, ax = plt.subplots(1, 3, figsize=[12, 12])
        ax[0].set_title("Original")
        ax[0].imshow(img, cmap='gray')
        ax[0].axis('off')
        ax[1].set_title("Apply Mask on Original")
        ax[1].imshow(cropped_image * mask, cmap='gray')
        ax[1].axis('off')
        ax[2].set_title("Original windowed")
        ax[2].imshow(set_manual_window(cropped_image * mask, 30, 150), cmap='gray')
        ax[2].axis('off')
        plt.show()
    
    
    result = cropped_image * mask
    del regions
    del labels
    del cropped_image
    del mask
    return result.astype('float64')


def img_intensity_histo(img_path):
    imgs_to_process = get_pixels_hu(img_path).astype(np.float64) 
    plt.hist(imgs_to_process.flatten(), bins=50, color='c')
    plt.xlabel("Hounsfield Units (HU)")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
class DataGenerator(keras.utils.Sequence):

    def __init__(self, list_IDs, labels=None, batch_size=1, img_size=(512, 512, 3), 
                 img_dir=train_images_dir, *args, **kwargs):

        self.list_IDs = list_IDs
        self.labels = labels
        self.batch_size = batch_size
        self.img_size = img_size
        self.img_dir = img_dir
        self.shuffle = False
        self.on_epoch_end()

    def __len__(self):
        return int(ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        indices = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        list_IDs_temp = [self.list_IDs[k] for k in indices]
        if self.labels is not None:
            X, Y = self.__data_generation(list_IDs_temp)
            return X, Y
        else:
            X = self.__data_generation(list_IDs_temp)
            return X
        
        
    def on_epoch_end(self):
#         if self.labels is not None: # for training phase we undersample and shuffle
#             # keep probability of any=0 and any=1
#             keep_prob = self.labels.iloc[:, 0].map({0: 0.35, 1: 0.5})
#             keep = (keep_prob > np.random.rand(len(keep_prob)))
#             self.indices = np.arange(len(self.list_IDs))[keep]
#             np.random.shuffle(self.indices)
#         else:
        self.indices = np.arange(len(self.list_IDs))
        if self.shuffle == True:
            np.random.shuffle(self.indices)

            
            
    def __data_generation(self, list_IDs_temp):
        X = np.empty((self.batch_size, *self.img_size))
        
        if self.labels is not None: # training phase
            Y = np.empty((self.batch_size, 6), dtype=np.float32)
            for i, ID in enumerate(list_IDs_temp):
                X[i,] = read_and_prep_images([ID+".dcm"], self.img_size, pixels_out=True)
                Y[i,] = self.labels.loc[ID].values

            return X, Y
        else: # test phase
            print('read and prep on test images')
            for i, ID in enumerate(list_IDs_temp):
                X[i,] = read_and_prep_images([ID+".dcm"], self.img_size,pixels_out=True,folder=self.img_dir)
            
            return X


In [ ]:
from keras import backend as K

def weighted_log_loss(y_true, y_pred):
    """
    Can be used as the loss function in model.compile()
    ---------------------------------------------------
    """
    
    class_weights = np.array([2., 1., 1., 1., 1., 1.])
    
    eps = K.epsilon()
    
    y_pred = K.clip(y_pred, eps, 1.0-eps)

    out = -(         y_true  * K.log(      y_pred) * class_weights
            + (1.0 - y_true) * K.log(1.0 - y_pred) * class_weights)
    
    return K.mean(out, axis=-1)


def _normalized_weighted_average(arr, weights=None):
    """
    A simple Keras implementation that mimics that of 
    numpy.average(), specifically for this competition
    """
    
    if weights is not None:
        scl = K.sum(weights)
        weights = K.expand_dims(weights, axis=1)
        return K.sum(K.dot(arr, weights), axis=1) / scl
    return K.mean(arr, axis=1)


def weighted_loss(y_true, y_pred):
    """
    Will be used as the metric in model.compile()
    ---------------------------------------------
    
    Similar to the custom loss function 'weighted_log_loss()' above
    but with normalized weights, which should be very similar 
    to the official competition metric:
        https://www.kaggle.com/kambarakun/lb-probe-weights-n-of-positives-scoring
    and hence:
        sklearn.metrics.log_loss with sample weights
    """
    
    class_weights = K.variable([2., 1., 1., 1., 1., 1.])
    
    eps = K.epsilon()
    
    y_pred = K.clip(y_pred, eps, 1.0-eps)

    loss = -(        y_true  * K.log(      y_pred)
            + (1.0 - y_true) * K.log(1.0 - y_pred))
    
    loss_samples = _normalized_weighted_average(loss, class_weights)
    
    return K.mean(loss_samples)


def weighted_log_loss_metric(trues, preds):
    """
    Will be used to calculate the log loss 
    of the validation set in PredictionCheckpoint()
    ------------------------------------------
    """
    class_weights = [2., 1., 1., 1., 1., 1.]
    
    epsilon = 1e-7
    
    preds = np.clip(preds, epsilon, 1-epsilon)
    loss = trues * np.log(preds) + (1 - trues) * np.log(1 - preds)
    loss_samples = np.average(loss, axis=1, weights=class_weights)

    return - loss_samples.mean()


In [ ]:
class PredictionCheckpoint(keras.callbacks.Callback):
    
    def __init__(self, test_df, valid_df, 
                 test_images_dir=test_images_dir, 
                 valid_images_dir=train_images_dir, 
                 batch_size=32, input_size=(224, 224, 3)):
        
        self.test_df = test_df
        self.valid_df = valid_df
        self.test_images_dir = test_images_dir
        self.valid_images_dir = valid_images_dir
        self.batch_size = batch_size
        self.input_size = input_size
        
    def on_train_begin(self, logs={}):
        self.test_predictions = []
        self.valid_predictions = []
        
    def on_epoch_end(self,batch, logs={}):
        self.test_predictions.append(
            self.model.predict_generator(
                DataGenerator(self.test_df.index, None, self.batch_size, self.input_size, self.test_images_dir), verbose=2)[:len(self.test_df)])
                # Commented out to save time
#         self.valid_predictions.append(
#             self.model.predict_generator(
#                 DataGenerator(self.valid_df.index, None, self.batch_size, self.input_size, self.valid_images_dir), verbose=2)[:len(self.valid_df)])
        
#         print("validation loss: %.4f" %
#               weighted_log_loss_metric(self.valid_df.values, 
#                                    np.average(self.valid_predictions, axis=0, 
#                                               weights=[2**i for i in range(len(self.valid_predictions))])))
        
        # here you could also save the predictions with np.save()

In [ ]:

class MyDeepModel:
    
    def __init__(self, engine, input_dims, batch_size=5, num_epochs=4, learning_rate=1e-3, 
                 decay_rate=1.0, decay_steps=1, weights='imagenet', verbose=1):
        
        self.engine = engine
        self.input_dims = input_dims
        self.batch_size = batch_size
        self.num_epochs = num_epochs
        self.learning_rate = learning_rate
        self.decay_rate = decay_rate
        self.decay_steps = decay_steps
        self.weights = weights
        self.verbose = verbose
        self._build()

    def _build(self):
        
    
        engine = self.engine(include_top=False, weights=self.weights, input_shape=(*self.input_dims[:2], 3),
                             backend = keras.backend, layers = keras.layers,
                             models = keras.models, utils = keras.utils)
        

        x = keras.layers.GlobalAveragePooling2D(name='avg_pool')(engine.output)
        x = keras.layers.Dropout(0.2)(x)
        x = keras.layers.Dense(keras.backend.int_shape(x)[1], activation="relu", name="dense_hidden_1")(x)
        x = keras.layers.Dropout(0.1)(x)
        out = keras.layers.Dense(6, activation="sigmoid", name='dense_output')(x)

        self.model = keras.models.Model(inputs=engine.input, outputs=out)

        self.model.compile(loss=weighted_log_loss, optimizer=keras.optimizers.Adam(), metrics=[weighted_loss])
    
    def fit_and_predict_subset(self, train_df, test_df, subsetNumber):
        print('fit and predict subset')
        test_df_sub = test_df.sample(subsetNumber)
        sample_df = train_df.sample(subsetNumber)
        ss = ShuffleSplit(n_splits=1, test_size=0.1, random_state=42).split(sample_df.index)
        train_idx, valid_idx = next(ss)
        model.fit_and_predict(sample_df.iloc[train_idx], sample_df.iloc[valid_idx], test_df_sub)
       

    def fit_and_predict(self, train_df, valid_df, test_df):
        
        # callbacks
        print('Fit and predict')
        pred_history = PredictionCheckpoint(test_df, valid_df, input_size=self.input_dims)
        print('pred_history done')
        checkpointer = keras.callbacks.ModelCheckpoint(filepath='%s-{epoch:02d}.hdf5' % self.engine.__name__, verbose=1, save_weights_only=True, save_best_only=False)
        print('checkpointer done')
        scheduler = keras.callbacks.LearningRateScheduler(lambda epoch: self.learning_rate * pow(self.decay_rate, floor(epoch / self.decay_steps)))
        print('scheduler done')
        
        self.model.fit_generator(
            DataGenerator(
                train_df.index, 
                train_df, 
                self.batch_size, 
                self.input_dims, 
                train_images_dir
            ),
            epochs=self.num_epochs,
            verbose=self.verbose,
            use_multiprocessing=True,
            workers=4,
            #callbacks=[scheduler]
            callbacks=[pred_history, scheduler]
        )
        
        return pred_history
    
    def predict(self,test_df):
        preds = self.model.predict_generator(DataGenerator(
                test_df.index, 
                test_df, 
                self.batch_size, 
                self.input_dims, 
                test_images_dir
            ),verbose=self.verbose,steps = 1)
        return preds
        
    def save(self, path):
        self.model.save_weights(path)
    
    def load(self, path):
        self.model.load_weights(path)

In [ ]:
def read_testset(filename="../input/rsna-intracranial-hemorrhage-detection/stage_2_sample_submission.csv"):
    
    df = pd.read_csv(filename)
    
    # Extract Id image form the ID column
    df['Image'] = df['ID'].apply(lambda st:"ID_" + st.split('_')[1])
    
    # Extract the subtype of hymorrage from the ID column
    df["Diagnosis"] =  df['ID'].apply(lambda st: st.split('_')[2])

    # Dropping the ID column
    df = df[['Image',"Diagnosis",'Label']]
    
    # Drop the duplicates
    df = df.drop_duplicates(subset=['Image','Diagnosis'], keep='first')
    
    ## Seting a multi indexing and pivot the table on dignosis
    df = df.set_index(['Image', 'Diagnosis']).unstack(level=-1)
    
    return df

def read_trainset(filename="../input/rsna-intracranial-hemorrhage-detection/stage_2_train.csv"):
    df = pd.read_csv(filename)
     
    # Extract Id image form the ID column
    df['Image'] = df['ID'].apply(lambda st:"ID_" + st.split('_')[1])
    
    # Extract the subtype of hymorrage from the ID column
    df["Diagnosis"] =  df['ID'].apply(lambda st: st.split('_')[2])
    
        
    # Dropping the ID column
    df = df[['Image',"Diagnosis",'Label']]
    
    # Drop the duplicates
    df = df.drop_duplicates(subset=['Image','Diagnosis'], keep='first')

    ## Seting a multi indexing and pivot the table on dignosis
    df = df.set_index(['Image', 'Diagnosis']).unstack(level=-1)
    return df

    
test_df = read_testset()
train_df = read_trainset()

In [ ]:
train_df.shape

In [ ]:
test_df.shape

In [ ]:

# obtain model
model = MyDeepModel(engine=InceptionV3, input_dims=(224, 224, 3), batch_size=32, learning_rate=5e-4,
                    num_epochs=4, decay_rate=0.8, decay_steps=1, weights="imagenet", verbose=1)


In [ ]:
sample_test_df = test_df.sample(32)
preds = model.predict(sample_test_df)

In [ ]:
preds

In [ ]:
# obtain test + validation predictions (history.test_predictions, history.valid_predictions)
sample_test_df = test_df.sample(50)
import warnings

with warnings.catch_warnings():
    warnings.simplefilter('error')
    #history = model.fit_and_predict(train_df.iloc[train_idx], train_df.iloc[valid_idx], test_df)
    history = model.fit_and_predict_subset(train_df, test_df, 10)

In [ ]:
# model.save("20191102-2315.weights")

In [ ]:
#test_df1 = test_df
preds = model.model.predict_generator(DataGenerator(test_df.index, None, 32, (224,224,3), test_images_dir), verbose=2)

In [ ]:
test_df = read_testset()

In [ ]:
print(test_df.head())
print(test_df.shape)
#preds.head()
print(len(history.test_predictions))
#print(history.test_predictions[0])
print(len(np.average(history.test_predictions, axis=0, weights=[0, 2, 4, 6]) ))

In [ ]:
#print(preds)
test_df.iloc[:, :] = np.average(history.test_predictions, axis=0, weights=[0, 2, 4, 6]) # let's do a weighted average for epochs (>1)

#test_df.iloc[:, :] = preds[0:78545]

test_df = test_df.stack().reset_index()

test_df.insert(loc=0, column='ID', value=test_df['Image'].astype(str) + "_" + test_df['Diagnosis'])

test_df = test_df.drop(["Image", "Diagnosis"], axis=1)



In [ ]:
test_df.to_csv('submission_with_history.csv', index=False)